In [2]:
#test1
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# D2Q9 discrete Velocity Vectors: [cx, cy]

c = np.array([
    [0,0], # 0: rest
    [1,0], #1: east
    [0,1], #2: north
    [-1,0], #3: west
    [0,-1], #4: south
    [1,1], #5: northeast
    [-1,1], #6: northwest
    [-1,-1], #7: southwest
    [1,-1]  #8: southeast
])

c

array([[ 0,  0],
       [ 1,  0],
       [ 0,  1],
       [-1,  0],
       [ 0, -1],
       [ 1,  1],
       [-1,  1],
       [-1, -1],
       [ 1, -1]])

In [4]:
# D2Q9 weights 
# hosen so that the D2Q9 lattice correctly reproduces the isotropic behaviour needed to recover incompressible Navier–Stokes flow at low Mach number

w = np.array([
    4/9, #0: rest
    1/9, #1: east
    1/9, #2: north
    1/9, #3: west
    1/9, #4: south
    1/36, #5: northeast
    1/36, #6: northwest
    1/36, #7: southwest
    1/36  #8: southeast
])
w
print("sum of weights = ", np.sum(w))

sum of weights =  1.0


In [5]:
# define popilations f_i

# One lattice node: nine directional populations
f = np.array([
    4/9,    # f0: rest
    1/9,    # f1: east
    1/9,    # f2: north
    1/9,    # f3: west
    1/9,    # f4: south
    1/36,   # f5: north-east
    1/36,   # f6: north-west
    1/36,   # f7: south-west
    1/36,   # f8: south-east
])

f
print("Original f:")
print(f)
print("Shape:", f.shape)

print("\nf[:, None]:")
print(f[:, None])
print("Shape:", f[:, None].shape)

Original f:
[0.44444444 0.11111111 0.11111111 0.11111111 0.11111111 0.02777778
 0.02777778 0.02777778 0.02777778]
Shape: (9,)

f[:, None]:
[[0.44444444]
 [0.11111111]
 [0.11111111]
 [0.11111111]
 [0.11111111]
 [0.02777778]
 [0.02777778]
 [0.02777778]
 [0.02777778]]
Shape: (9, 1)


In [6]:
# Density = sum of populations at a lattice node
rho = np.sum(f)

# momentum = sum of populations multiplied by their respective velocity vectors
momentum = np.sum(f[:, None] * c, axis=0)
# [:,none] - the none is used to add another dimension - is used to convert the 1D array f into a 2D column vector so that it can be multiplied element-wise with the 2D array c. The result is a 2D array where each row corresponds to a population and its associated velocity vector. The np.sum(..., axis=0) then sums over the rows to give the total momentum in each direction (x and y).

# axis =0 - specifies that the summation should be performed along the first axis (rows) of the resulting 2D array. This means that we are summing the contributions of all populations to get the total momentum in each direction.

# note we are not doin matrix multiplication here, we are doing element-wise multiplication and then summing the results to get the total momentum in each direction.

# average velocity = momentum / density
u = momentum / rho

print("Density:", rho)
print("Momentum:", momentum)
print("Velocity:", u)

Density: 1.0
Momentum: [0. 0.]
Velocity: [0. 0.]


In [7]:
#create a tiny, controlled rightward flow at one lattice node.

#We will increase the east-going population and decrease the west-going population by the same amount. This preserves density, but creates positive x-momentum.

# Start again from the stationary equilibrium state
# Make a new array named f by copying w.
# w contains the equilibrium D2Q9 weights.
# .copy() is important: it gives f its own separate values,
# so changing f later does not alter w.
f = w.copy()


# Store 0.02 in a variable named delta.
# This is the amount of population we will transfer
# from the west-going direction to the east-going direction.
delta = 0.02


# f[1] is the east-going population because c[1] = [1, 0].
# Add delta, so slightly more population travels east.
f[1] = f[1] + delta


# f[3] is the west-going population because c[3] = [-1, 0].
# Subtract the same delta, so slightly less population travels west.
# Adding and subtracting the same amount preserves total density.
f[3] = f[3] - delta


# print() displays information but does not change any values.
# This displays the complete modified array f.
print("Modified f:", f)


# np.sum(f) adds all nine populations f[0] through f[8].
# Store that total in rho, the local density.
rho = np.sum(f)


# Display the density. It should still be 1.0.
print("Density:", rho)

Modified f: [0.44444444 0.13111111 0.11111111 0.09111111 0.11111111 0.02777778
 0.02777778 0.02777778 0.02777778]
Density: 0.9999999999999999


In [8]:
# Create a NumPy array with two zeros:
# first position = total x-momentum
# second position = total y-momentum
momentum = np.array([0.0, 0.0])


# range(9) produces the direction indices:
# 0, 1, 2, 3, 4, 5, 6, 7, 8.
# The loop visits every D2Q9 direction once.
for i in range(9):

    # f[i] is the population travelling in direction i.
    # c[i] is that direction's vector [cx, cy].
    # f[i] * c[i] calculates this population's x- and y-momentum contribution.
    #
    # Example for i = 1:
    # f[1] * c[1] = f[1] * [1, 0]
    #
    # Add that contribution to the running total momentum.
    momentum = momentum + f[i] * c[i]


# Momentum equals rho * velocity.
# Divide both x- and y-momentum by density rho
# to calculate the local velocity vector u = [ux, uy].
u = momentum / rho


# Display the final momentum vector [x-momentum, y-momentum].
print("Momentum:", momentum)


# Display the final velocity vector [ux, uy].
print("Velocity:", u)

Momentum: [0.04 0.  ]
Velocity: [0.04 0.  ]


In [9]:
# Set the density we want at this lattice node.
# In lattice units, rho = 1.0 is the usual convenient starting value.
rho_target = 1.0

# Create a two-element NumPy array named u_target, initialized to zeros.
# This represents the target velocity vector [ux, uy] at this lattice node. 
# Both components are initially set to zero, indicating no flow in either direction.

u_target = np.array([0.0, 0.0])

# Multiply every D2Q9 weight by the target density to get the equilibrium populations.
# This is done using element-wise multiplication of the weight array w with the scalar rho_target.
#  f_eq[0], f_eq[1], ..., f_eq[8] are the equilibrium populations corresponding to each of the nine discrete velocity directions in the D2Q9 model.
f_eq = rho_target * w

# Display the nine equilibrium populations.
print("Equilibrium populations:", f_eq)
#note that the original equation of F_eq = rho * w * (1 + 3 * (c @ u) + 9/2 * (c @ u)**2 - 3/2 * (u @ u)) is a more general form that accounts for non-zero velocities. In this case, since we are setting the target velocity to zero, the simplified version f_eq = rho_target * w is sufficient to represent the equilibrium state at rest.
# in current case we have chosen ideal values u = [0,0] where everything els becomes 1



# Add all nine equilibrium populations.
# This checks whether they recover the density we prescribed.
rho_check = np.sum(f_eq)


# Display the recovered density.
print("Recovered density:", rho_check)



Equilibrium populations: [0.44444444 0.11111111 0.11111111 0.11111111 0.11111111 0.02777778
 0.02777778 0.02777778 0.02777778]
Recovered density: 1.0


In [10]:
# Set the density that we want at this one lattice node.
rho_target = 1.0


# Set the desired macroscopic velocity [ux, uy].
# ux = 0.04 means a small flow to the right.
# uy = 0.0 means no upward or downward flow.
u_target = np.array([0.04, 0.0])


# In D2Q9 lattice units, the square of the lattice sound speed is fixed.
# cs^2 = 1/3.
cs_squared = 1 / 3


# c @ u_target performs a dot product for every direction i.
# Result i is: c[i, 0] * ux + c[i, 1] * uy.
# This gives the velocity component aligned with each D2Q9 direction.
c_dot_u = c @ u_target


# np.dot(u_target, u_target) calculates ux^2 + uy^2.
# This is the squared magnitude of the macroscopic velocity.
u_squared = np.dot(u_target, u_target)


# Use the complete second-order D2Q9 equilibrium distribution:
#
# f_eq[i] = w[i] * rho * (
#     1
#     + (c[i] dot u) / cs^2
#     + (c[i] dot u)^2 / (2 * cs^4)
#     - (u dot u) / (2 * cs^2)
# )
#
# cs_squared**2 means (cs^2)^2 = cs^4.
f_eq = w * rho_target * (
    1
    + c_dot_u / cs_squared
    + (c_dot_u ** 2) / (2 * cs_squared ** 2)
    - u_squared / (2 * cs_squared)
)


# Display the nine equilibrium populations.
print("Equilibrium populations:", f_eq)


# Recover density by adding all nine populations.
rho_check = np.sum(f_eq)


# Start the total momentum at [0, 0].
momentum_check = np.array([0.0, 0.0])


# Visit every D2Q9 direction.
for i in range(9):

    # Add equilibrium population i multiplied by its direction vector.
    momentum_check = momentum_check + f_eq[i] * c[i]


# Divide momentum by density to recover velocity.
u_check = momentum_check / rho_check


# Verify that the equilibrium populations reproduce our prescribed quantities.
print("Recovered density:", rho_check)
print("Recovered velocity:", u_check)

Equilibrium populations: [0.44337778 0.12497778 0.11084444 0.09831111 0.11084444 0.03124444
 0.02457778 0.02457778 0.03124444]
Recovered density: 0.9999999999999999
Recovered velocity: [4.00000000e-02 3.46944695e-18]


In [ ]:
# Lattice spacing: one lattice-node distance.
dx = 1.0

# Lattice time step: one simulation tick.
dt = 1.0

# Lattice speed: dx / dt = 1.
lattice_speed = dx / dt

# D2Q9 lattice speed of sound squared.
cs_squared = lattice_speed**2 / 3

In [11]:
# Set the relaxation time, tau.
# For the basic BGK model, tau must be greater than 0.5.
# We use 0.8: a common safe demonstration value.
tau = 0.8


# Create a deliberately non-equilibrium distribution.
# .copy() creates a separate array, so f_current can change
# without changing f_eq.
f_current = f_eq.copy()


# Add a small amount to the east-going population, f1.
# This makes f_current different from the target equilibrium.
f_current[1] = f_current[1] + 0.01


# Subtract the same amount from the west-going population, f3.
# Total density remains unchanged, but the distribution is now
# more strongly right-moving than the equilibrium state.
f_current[3] = f_current[3] - 0.01


# Calculate how far each current population is from equilibrium.
# A positive value means that direction currently has too much population.
# A negative value means that direction currently has too little population.
non_equilibrium_part = f_current - f_eq


# Divide the difference by tau.
# This sets how much of the difference collision removes this time step.
collision_change = non_equilibrium_part / tau


# Subtract the collision change from the current population.
# This moves every population toward f_eq.
# f_post_collision is often written as f_star, f*, in equations.
f_post_collision = f_current - collision_change


# Print each stage so you can compare the nine populations.
print("Equilibrium f_eq:       ", f_eq)
print("Current f:              ", f_current)
print("Difference (f - f_eq):  ", non_equilibrium_part)
print("After collision f_star: ", f_post_collision)

Equilibrium f_eq:        [0.44337778 0.12497778 0.11084444 0.09831111 0.11084444 0.03124444
 0.02457778 0.02457778 0.03124444]
Current f:               [0.44337778 0.13497778 0.11084444 0.08831111 0.11084444 0.03124444
 0.02457778 0.02457778 0.03124444]
Difference (f - f_eq):   [ 0.    0.01  0.   -0.01  0.    0.    0.    0.    0.  ]
After collision f_star:  [0.44337778 0.12247778 0.11084444 0.10081111 0.11084444 0.03124444
 0.02457778 0.02457778 0.03124444]


In [12]:
print("f_eq[1], f_current[1], f_post_collision[1]:")
print(f_eq[1], f_current[1], f_post_collision[1])

f_eq[1], f_current[1], f_post_collision[1]:
0.1249777777777778 0.1349777777777778 0.1224777777777778


In [13]:
# Add all nine current populations to recover the current density.
rho_current = np.sum(f_current)


# Create x- and y-momentum totals, initially both zero.
momentum_current = np.array([0.0, 0.0])


# Visit each of the nine D2Q9 directions.
for i in range(9):

    # Add population i multiplied by its direction vector.
    # This builds the total momentum vector [rho*ux, rho*uy].
    momentum_current = momentum_current + f_current[i] * c[i]


# Divide momentum by density to recover the current velocity.
u_current = momentum_current / rho_current


# Calculate c_i dot u_current for all nine directions.
c_dot_u_current = c @ u_current


# Calculate ux^2 + uy^2 for the current velocity.
u_squared_current = np.dot(u_current, u_current)


# Build the equilibrium distribution that matches f_current's
# own density and velocity.
f_eq_current = w * rho_current * (
    1
    + c_dot_u_current / cs_squared
    + (c_dot_u_current ** 2) / (2 * cs_squared ** 2)
    - u_squared_current / (2 * cs_squared)
)


# Apply BGK collision toward this matching equilibrium.
f_post_collision = f_current - (f_current - f_eq_current) / tau


# Display the macroscopic state used to construct equilibrium.
print("Current density:", rho_current)
print("Current velocity:", u_current)

Current density: 0.9999999999999999
Current velocity: [6.00000000e-02 3.46944695e-18]


In [14]:
# Add all nine post-collision populations.
rho_after = np.sum(f_post_collision)


# Start the post-collision momentum total at zero.
momentum_after = np.array([0.0, 0.0])


# Add the momentum contribution from every post-collision population.
for i in range(9):
    momentum_after = momentum_after + f_post_collision[i] * c[i]


# Convert the post-collision momentum into velocity.
u_after = momentum_after / rho_after


# Compare state before and after collision.
print("Density before collision:", rho_current)
print("Density after collision: ", rho_after)

print("Velocity before collision:", u_current)
print("Velocity after collision: ", u_after)

Density before collision: 0.9999999999999999
Density after collision:  0.9999999999999998
Velocity before collision: [6.00000000e-02 3.46944695e-18]
Velocity after collision:  [6.0000000e-02 6.9388939e-18]
